# 02 — Comparing forecasting models

This notebook compares Ridge, Random Forest, and gradient boosting with a straightforward baseline: predict that the next close will equal the last one. XGBoost can be added if installed.

The training, validation, and test periods follow chronological target dates. Model selection is based on validation RMSE. The printed test results are useful for comparison, but should not be used for another round of model selection. All errors are measured in S&P 500 index points.

In [ ]:
from pathlib import Path
from sp500_research import load_prices, download_sp500, build_supervised_frame, chronological_split
DATA_FILE = Path('../data/sp500_ohlcv.csv')
if DATA_FILE.exists():
    prices = load_prices(DATA_FILE)
else:
    # Download a sample if no local CSV is available (internet required).
    prices = download_sp500(start='2000-01-01', end='2025-01-01')
print('Data coverage:', prices.index.min().date(), 'to', prices.index.max().date(), len(prices), 'daily rows')

In [ ]:
from sp500_research.evaluation import train_and_compare
supervised=build_supervised_frame(prices)
parts=chronological_split(supervised)
# Set include_xgboost=True after installing xgboost to add that model.
score_table, chosen, fitted = train_and_compare(parts, include_xgboost=False)
print('Best validation RMSE:',chosen)
print(score_table.sort_values(['split','RMSE']).round(3).to_string(index=False))

In [ ]:
from sp500_research.features import model_features
from sp500_research.evaluation import regression_metrics
cols=model_features(supervised)
if chosen in fitted:
    actual=parts.test['target_next_close']
    prediction=fitted[chosen].predict(parts.test[cols])
else:
    actual=parts.test['target_next_close']
    prediction=parts.test['close_t']
print('Selected test metrics (index points):',regression_metrics(actual,prediction))

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(9,4))
ax.plot(parts.test.target_date,actual,label='Next close: actual',linewidth=1)
ax.plot(parts.test.target_date,prediction,label='Chosen validation model',linewidth=1,alpha=.8)
ax.set(xlabel='Target date', ylabel='Index points', title='Test period (new run only)')
ax.legend();plt.tight_layout();plt.show()

### Reading the results

Price levels often change slowly relative to their absolute value, so the last-close baseline is important. A more complex model needs to improve on that reference, not merely produce a visually close prediction. These are results from a new run, not the metrics reported in the 2025 paper.